In [ ]:
import altair as alt
from google.colab import files
uploaded = files.upload()

Saving CS 1377 Final Visualizations - Sheet1.csv to CS 1377 Final Visualizations - Sheet1.csv


In [ ]:
import pandas as pd
dt = pd.read_csv('CS 1377 Final Visualizations - Sheet1.csv')
print(dt)

      MCQ Type  Mean Score
0  Note-Taking     0.69530
1     Combined     0.59895


In [ ]:
chart = alt.Chart(dt).mark_bar().encode(
    # Sort the x-axis based on the y-axis values in descending order ('-y')
    x=alt.X('MCQ Type', sort='-y', title='Type of Test Taker'),
    y=alt.Y('Mean Score', title='Mean Score'),
    # Color the bars based on the fruit name
    color='MCQ Type'
).properties(
    title=alt.TitleParams(text='Scores of MCQ Group', fontSize=18),
    width=400,
    height=300
)
chart.show()

alt.Chart(...)

In [ ]:
from google.colab import files
uploaded_new = files.upload()

Saving CS 1377 Final Visualizations - Sheet1 (1).csv to CS 1377 Final Visualizations - Sheet1 (1).csv


In [ ]:
dt_new = pd.read_csv('CS 1377 Final Visualizations - Sheet1 (1).csv')
# 2. Reshape the data to "long" format for Altair
# This turns the two average columns into 'Test Type' and 'Average Score' columns
melted_data = dt_new.melt(
    id_vars='Group',
    var_name='Test Type',
    value_name='Average Score'
)

# 3. Build the grouped bar chart
chart_new = alt.Chart(melted_data).mark_bar().encode(
    # x represents the main groups on the axis
    x=alt.X('Group:N', title='Study Group', axis=alt.Axis(labelAngle=0)),

    # y represents the height of the bars
    y=alt.Y('Average Score:Q', title='Test Average'),

    # color separates the bars within each group by test type
    color=alt.Color('Test Type:N', title='Test Type'),

    # xOffset places the colored bars side-by-side instead of stacking them
    xOffset='Test Type:N'
).properties(
    # Keeping the bigger title from your previous request!
    title=alt.TitleParams(text='Test Averages by Study Group', fontSize=16),
    width=400,
    height=300
).configure_legend(
    # Increase the legend text size here
    titleFontSize=16,
    labelFontSize=14,
    symbolSize=200 # Optional: Makes the little color squares bigger to match the text
)

chart_new.show()

alt.Chart(...)

In [ ]:
from google.colab import files
uploaded_new = files.upload()


Saving CS 1377 Final Visualizations - Sheet1 (2).csv to CS 1377 Final Visualizations - Sheet1 (2).csv


In [ ]:
import pandas as pd
import altair as alt
data = [
    {'Question': 1, 'Type': 'MC', 'Percentage Correct': 0.7895},
    {'Question': 2, 'Type': 'MC', 'Percentage Correct': 0.3680},
    {'Question': 3, 'Type': 'MC', 'Percentage Correct': 0.6320},
    {'Question': 4, 'Type': 'MC', 'Percentage Correct': 0.9470},
    {'Question': 5, 'Type': 'MC', 'Percentage Correct': 0.5260},
    {'Question': 6, 'Type': 'MC', 'Percentage Correct': 0.8420},
    {'Question': 7, 'Type': 'MC', 'Percentage Correct': 0.7890},

    {'Question': 8, 'Type': 'FRQ', 'Percentage Correct': 0.2105},
    {'Question': 9, 'Type': 'FRQ', 'Percentage Correct': 0.7544},
    {'Question': 10, 'Type': 'FRQ', 'Percentage Correct': 0.7807},
    {'Question': 11, 'Type': 'FRQ', 'Percentage Correct': 0.7456},
]

df = pd.DataFrame(data)

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('Question:O', title='Question Number'),
    y=alt.Y('Percentage Correct:Q', axis=alt.Axis(format='%'), title='Percentage Correct'),
    xOffset='Type:N',
    color=alt.Color('Type:N', title='Question Type')
).properties(
    title='Percentage Correct by Question Type (MC vs. FRQ)',
    width=500,
    height=300
)

chart.show()


alt.Chart(...)

In [ ]:
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import numpy as np
import math
from statsmodels.stats.power import FTestAnovaPower


## GENERAL RESULTS ##

# scores per group (frq, mcq, no_questions)
# one value in a list represents the final score for one student in that group
frq_group_scores = [0.6563, 0.7031, 0.8444, 0.4375, 0.6875, 0.7188, 0.75]
mcq_group_scores = [0.75, 0.875, 0.2344, 0.6406, 0.5156, 0.5781]
no_question_group_scores = [0.2813, 0.5, 0.6094, 0.25, 0.9375, 0.9375]

# convert to dataframe for analysis
data_dict = {
    'score': frq_group_scores + mcq_group_scores + no_question_group_scores,
    'group': ['FRQ']*len(frq_group_scores) + ['MCQ']*len(mcq_group_scores) + ['None']*len(no_question_group_scores)
}
df = pd.DataFrame(data_dict)

# basic stats
print("-- Basic Statistics by Group --")
stats_df = df.groupby('group')['score'].agg(['mean', 'std', 'count', 'min', 'max', 'median'])
stats_df.columns = ['Mean', 'Std Dev', 'N', 'Min', 'Max', 'Median']
print(stats_df.to_string())
print("\n")

# One-Way ANOVA
f_stat, p_val_anova = stats.f_oneway(frq_group_scores, mcq_group_scores, no_question_group_scores)
print(f"-- One-Way ANOVA --")
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {p_val_anova:.4f}")
print("\n")

# Pairwise T-Tests
def run_ttest(group1, group2, name1, name2):
    t_stat, p_val = stats.ttest_ind(group1, group2, equal_var=False)
    print(f"{name1} vs {name2}: t={t_stat:.4f}, p-value={p_val:.4f}")

print("-- Pairwise T-Test Results (Welch's) --")
run_ttest(frq_group_scores, mcq_group_scores, "FRQ", "MCQ")
run_ttest(frq_group_scores, no_question_group_scores, "FRQ", "None")
run_ttest(mcq_group_scores, no_question_group_scores, "MCQ", "None")
print("\n")

# Tukey HSD (if ANOVA is significant)
if p_val_anova < 0.05:
    print("--Tukey HSD--")
    tukey = pairwise_tukeyhsd(endog=df['score'], groups=df['group'], alpha=0.05)
    print(tukey)
    print("\n")

## POWER ANALYSIS (FRQ vs. MCQ vs. None) ##

all_groups = [frq_group_scores, mcq_group_scores, no_question_group_scores]
all_scores = frq_group_scores + mcq_group_scores + no_question_group_scores
tot_mean = np.mean(all_scores)
k_groups = len(all_groups)

ssb = 0
ssw = 0

for group in all_groups:
    group_mean = np.mean(group)
    n_group = len(group)
    ssb += n_group * (group_mean - tot_mean)**2
    ssw += np.sum((np.array(group)-group_mean)**2)
sst = ssb+ssw
eta_sq = ssb / sst
coh_f = math.sqrt(eta_sq / (1-eta_sq))
pow_analysis = FTestAnovaPower()
tot_sample_size = pow_analysis.solve_power(effect_size = coh_f, k_groups = k_groups, alpha = 0.05, power = 0.80)
n_per_group = math.ceil (tot_sample_size/k_groups)
print("--Power Analysis--")
print(f"-> Total Required N: {math.ceil(tot_sample_size)} participants")
print(f"Required N per group: {n_per_group} participants")

-- Basic Statistics by Group --
           Mean   Std Dev  N     Min     Max   Median
group                                                
FRQ    0.685371  0.124625  7  0.4375  0.8444  0.70310
MCQ    0.598950  0.219560  6  0.2344  0.8750  0.60935
None   0.585950  0.303574  6  0.2500  0.9375  0.55470


-- One-Way ANOVA --
F-statistic: 0.3892
p-value: 0.6838


-- Pairwise T-Test Results (Welch's) --
FRQ vs MCQ: t=0.8535, p-value=0.4193
FRQ vs None: t=0.7499, p-value=0.4798
MCQ vs None: t=0.0850, p-value=0.9341


--Power Analysis--
3
-> Total Required N: 202 participants
Required N per group: 68 participants
